In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path
import numpy as np
import malaya_speech
from malaya_speech.model.clustering import StreamingKMeans

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
2025-09-28 23:55:39.009892: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759103739.019163  115524 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759103739.023607  115524 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1759103739.028907  115524 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:175910373

In [2]:
df = pd.read_parquet('anv_data_ke-audio.parquet')

In [3]:
mapping = {}
for i in tqdm(range(len(df))):
    mapping[df['audio'].iloc[i]] = i
len(mapping)

100%|██████████| 455592/455592 [00:01<00:00, 267478.77it/s]


455592

In [4]:
from datasets import load_dataset

ds = load_dataset("malaysia-ai/Multilingual-TTS", 'anv_data_ke')

Generating train split: 100%|██████████| 455592/455592 [00:00<00:00, 2813088.08 examples/s]


In [5]:
ds = ds['train'].to_pandas()

In [8]:
audio_filename = ds['audio_filename'].tolist()
len(audio_filename)

455592

In [14]:
from collections import defaultdict

locales = defaultdict(list)
for f in audio_filename:
    l = f.split('_')[3]
    locales[l].append(f)

In [13]:
f.split('_')[3]

'luo'

In [26]:
import faiss

data = {}
for k, files in locales.items():
    d = 192
    index = faiss.IndexFlatL2(d)
    
    centroids = []
    
    def assign(x, threshold=0.1):
        if len(centroids) == 0:
            centroids.append(x)
            index.add(np.array([x], dtype=np.float32))
            return 0
        
        D, I = index.search(np.array([x], dtype=np.float32), 1)
        if D[0][0] > threshold:
            centroids.append(x)
            index.add(np.array([x], dtype=np.float32))
            return len(centroids)-1
        else:
            return I[0][0]
            
    for i in tqdm(range(len(files))):
        index_ = mapping[files[i]]
        v_f = f'anv_data_ke-audio/{index_}.npy'
        if not os.path.exists(v_f):
            continue
        try:
            v = np.load(v_f)
            data[files[i]] = assign(v)
        except Exception as e:
            pass

100%|██████████| 80181/80181 [00:17<00:00, 4499.52it/s]


In [27]:
len(data)

455592

In [28]:
rows = ds.to_dict(orient = 'records')
for i in range(len(rows)):
    s = data[rows[i]['audio_filename']]
    rows[i]['speaker'] = rows[i]['speaker'] + f'_{s}'

In [29]:
from datasets import Dataset

dataset = Dataset.from_list(rows)
dataset[0]

{'audio_filename': 'anv_data_ke_luo_audio/anv_data_ke-luo-train-unscripted-audios-train_unscripted_007_0.mp3',
 'text': "Olemoni otow, bathe konchiel otow. To mae [?] inyalo kel kod kute mawuotho, kata koso goyone yath e sama owinjore. To mae inyalo geng' kod yath ma igoyo dinwoya dinwoya mar mondo mi ogeng' kute manyalo molo yadhno, manyalo molo olemono ma kelne chandruok kabila no.",
 'speaker': 'anv_data_ke_luo_audio_0'}

In [31]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'anv_data_ke')

Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00,  7.00ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 51.9MB / 51.9MB, 6.49MB/s  
Processing Files (1 / 1): 100%|██████████| 51.9MB / 51.9MB, 6.33MB/s  
New Data Upload: 100%|██████████| 2.99MB / 2.99MB,  365kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:09<00:00,  9.47s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/f2ace9b5a2615f8142985e157b93b7689ff693c7', commit_message='Upload dataset', commit_description='', oid='f2ace9b5a2615f8142985e157b93b7689ff693c7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)